<a href="https://colab.research.google.com/github/Mitul-Marimuthu/deep-learning/blob/phase2/cat_dog_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Switch runtime to T4 GPU To ensure GPU availability.

In [2]:
!pip install torch torchvision matplotlib numpy -q

import torch
import torchvision
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.10.0+cu128
GPU available: True


In [4]:
# Download directly from Microsoft's URL (original source)
!wget -q https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q kagglecatsanddogs_5340.zip -d cats_dogs
!ls cats_dogs/PetImages

Cat  Dog


In [6]:
# Set up the dataset
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

# some images are corrupted --> remove them
import os
from PIL import Image

def remove_corrupted(folder):
    removed = 0
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        try:
            img = Image.open(fpath)
            img.verify()
        except:
            os.remove(fpath)
            removed += 1
    print(f"Removed {removed} corrupted images from {folder}")

remove_corrupted('cats_dogs/PetImages/Cat')
remove_corrupted('cats_dogs/PetImages/Dog')

# Define transforms - resize everything to 224 x 224, normalize
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
# specific mean/std values are ImageNet dataset statistics.
# use them because resnet was pretained on ImageNet
# normalizing them with the same values it was trained on
# makes transfer learning work much better

# Load dataset
# Automatically assigns labels based on folder names.
# Cat -> 0, Dog -> 1
full_dataset = datasets.ImageFolder('cats_dogs/PetImages', transform=transform)
print(f"Total images: {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}") # ['Cat', 'Dog']

# 80/20 train/val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


Removed 0 corrupted images from cats_dogs/PetImages/Cat
Removed 0 corrupted images from cats_dogs/PetImages/Dog
Total images: 24998
Classes: ['Cat', 'Dog']
Train batches: 625, Val batches: 157


In [9]:
# Build a CNN from scratch in PyTorch

import torch
import torch.nn as nn
import torch.nn.functional as F

# All PyTorch modules inherit from nn.Module
# handles parameter tracking, .to(device), saving/loading, and more
# parameters() method from autograd is built in
class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional layers
        # 3 input channels (RGB), 32 filters, each filter is 3x3
        # padding=1 keeps spatial dimensions the same after convolution
        # so only pooling shrinks them

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1) # 3 channels in (RGB), 32 filters out
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size = 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2) # halves spatial dimensions each time
        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        # After 3 pooling operations 224 -> 112 -> 56 -> 28
        # so feature map is 128 channels x 28 x 28
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 2) # 2 output classes: cat, dog

    def forward(self, x):
        # Block 1
        x = self.pool(F.relu(self.conv1(x))) # (batch, 32, 112, 112)

        # Block 2
        x = self.pool(F.relu(self.conv2(x))) # (batch, 64, 56, 56)

        # Block 3
        x = self.pool(F.relu(self.conv3(x))) # (batch, 128, 28, 28)

        # Flatten
        # flattens everything except the batch dimension,
        # same as Numpy's reshape but for tensors.
        # -1 tells pytorch to figure out that dimension
        # automatically
        x = x.view(x.size(0), -1) # (batch, 128*28*28)

        # Fully connected
        # no softmax needed because nn.CrossEntropyLoss applies
        # softmax internally, so you return raw logits.
        # adding softmax ourselves would apply it twice
        # and break training
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x) # raw logits, no softmax needed here

        return x

# initialize and move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CatDogCNN().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Total parameters: {total_params:,}")

# Verify forward pass works with a dummy batch
dummy = torch.randn(4, 3, 224, 224).to(device)
out = model(dummy)
print(f"Input shape: {dummy.shape}")
print(f"Output shape: {out.shape}") # expect (4, 2)

Device: cuda
Total parameters: 51,475,010
Input shape: torch.Size([4, 3, 224, 224])
Output shape: torch.Size([4, 2])


What is a CNN and how is it different?

In Project 1, every neuron was connected to every pixel — a 784-input image meant 784 weights per neuron. This is called a fully connected or dense layer. It works for MNIST because the images are tiny and simple, but it breaks down for real images:

A 224×224 RGB image has 224 × 224 × 3 = 150,528 pixels. A fully connected first layer with 512 neurons would need 150,528 × 512 = 77 million weights — just for one layer. That's slow, expensive, and it ignores something important:
Images have spatial structure. A cat's ear in the top-left corner looks the same as a cat's ear in the bottom-right corner. A fully connected layer has no concept of this — it treats every pixel as independent.

A CNN solves this with a filter (also called a kernel) — a small learnable grid of weights, say 3×3. Instead of connecting every neuron to every pixel, the filter slides across the image, computing a dot product at each position. The same filter is reused everywhere — so the "detect a curved edge" filter works whether the edge is at the top or bottom of the image. This is called weight sharing and it's the core idea of CNNs.
The result is:

- Far fewer parameters (32 filters × 3×3 weights = 288 weights vs millions)
- Spatial awareness — nearby pixels are processed together
- Translation invariance — the same feature detected anywhere in the image

` self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1) `

Breaking down each argument:

3 — number of input channels. RGB images have 3 (red, green, blue). The second conv layer takes 32 because that's how many channels conv1 outputs.

32 — number of filters to learn. Each filter is a different 3×3 detector — one might learn to detect horizontal edges, another vertical edges, another color gradients. You get 32 different feature maps out.

kernel_size=3 — each filter is a 3×3 grid of weights. It looks at a 3×3 patch of the image at a time.

padding=1 — adds a border of zeros around the image before convolving. Without it, a 3×3 filter on a 224×224 image produces a 222×222 output (you lose 1 pixel on each side). With padding=1 the output stays 224×224 — only pooling changes the spatial size.

This happens at every position across the image, producing one complete feature map per filter. With 32 filters you get 32 feature maps — 32 different views of the same image, each highlighting different patterns.

`
x = self.pool(F.relu(self.conv1(x)))   # 224 → 112
x = self.pool(F.relu(self.conv2(x)))   # 112 → 56
x = self.pool(F.relu(self.conv3(x)))   # 56  → 28
`

nn.MaxPool2d(2, 2) slides a 2×2 window across the feature map and keeps only the maximum value in each window, then moves 2 pixels (the stride). This halves both dimensions:

Why max? The maximum value in a region indicates whether a feature was detected anywhere in that patch. The exact location matters less than whether it was present at all. This gives the network a degree of position tolerance — a whisker slightly higher or lower still activates the same neuron.

Why do it at all? Three reasons:

- Reduces computation — smaller feature maps = fewer operations in later layers
- Increases receptive field — after pooling, each value in the next layer "sees" a larger region of the original image
- Reduces overfitting — less spatial precision means the network focuses on presence of features, not exact pixel positions

After three rounds of pooling, 224 → 112 → 56 → 28, each value in the final feature map has a receptive field covering a large chunk of the original image — the network is now reasoning about high-level structure, not individual pixels.
